# Train NEXUS Vision on a free GPU

This trains your own NEXUS Vision, the model NEXUS Live uses to see your screen and pick the next click or key,
and uploads it to a **private** repo on your Hugging Face account. Your laptop then downloads it once and runs it
offline, with no per-token cost.

It fine-tunes Qwen3-VL-2B-Instruct (Apache-2.0) on Windows screenshots from OS-Atlas (Apache-2.0) and, if you add
them, your own teach recordings, then tests the result against the base model on examples it never saw.

**On Kaggle (free, runs in the background):**
1. Settings (right panel): Accelerator **GPU T4 x2**, Internet **On** (Kaggle asks you to verify your phone once).
2. Add-ons > Secrets: add `HF_TOKEN` with a Hugging Face token that can write
   (https://huggingface.co/settings/tokens), and tick it for this notebook.
3. Fill in the settings cell below.
4. **Save Version > Save & Run All (Commit) > Save.** You can close the tab: it runs on Kaggle's side for a few hours,
   and the model lands in your Hugging Face repo (and in this notebook's Output tab).

**On Colab:** Runtime > Change runtime type > T4 GPU, add `HF_TOKEN` under the key icon (Secrets), fill in the
settings, then Runtime > Run all, and keep the tab open until it finishes.

## Settings

In [ ]:
REPO = "YOUR_NAME/nexus-vision"  # the model repo on your Hugging Face account; it is created private
REF = "main"                     # the nexus version to train with; use "claude/nexus-live" until that branch is merged
LIMIT = 3000                     # OS-Atlas elements to learn from (3000 takes a few hours on a free T4)
TEACH = ""                       # optional: your recordings as a dataset repo, such as YOUR_NAME/nexus-teach

In [ ]:
import os

# One graphics card: with two (Kaggle's T4 x2), training would split each batch across them, which a 4-bit model
# does not survive. This has to happen before anything loads torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
assert "/" in REPO and not REPO.startswith("YOUR_NAME/"), "Put your Hugging Face user name in REPO first."

In [ ]:
!pip -q install "nexus-ai[vision] @ git+https://github.com/nagu-io/nexus@{REF}"

## Hugging Face token

Read from the notebook's secrets. Never paste a token into a code cell.

In [ ]:
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

token = _secret("HF_TOKEN")
if not token:
    raise SystemExit("Add a secret named HF_TOKEN with a Hugging Face token that can write (see the steps at the top).")
os.environ["HF_TOKEN"] = token

from huggingface_hub import HfApi

account = HfApi().whoami()
print("Logged in to Hugging Face as", account["name"])

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit("No GPU: turn one on in the notebook settings (GPU T4 x2 on Kaggle, T4 on Colab).")
major, minor = torch.cuda.get_device_capability(0)
memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"{torch.cuda.get_device_name(0)}, {memory:.0f} GB, compute capability {major}.{minor}")

## Train, test and upload

In [ ]:
from nexus.live.build import main

args = ["--systems", "windows", "--limit", str(LIMIT), "--push", REPO,
        "--work", "/tmp/nexus_vision_work", "--out", "/kaggle/working/nexus_vision" if os.path.isdir("/kaggle") else "nexus_vision",
        "--batch-size", "2", "--grad-accum", "8", "--eval-limit", "150"]
if TEACH:
    args += ["--teach", TEACH]
main(args)

## On your laptop

```
pip install -e .[live,vision]
nexus vision pull YOUR_NAME/nexus-vision
nexus live --check
nexus live --voice
```